In [1]:
import sys
sys.path.append("..")  # utilsは1階層上にある

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from utils.models import LSTMGaussian,PortfolioLSTMModel
from utils.trainer import DreamingTrainer
from utils.datawindow import TorchDataWindow
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

In [2]:
# データ読み込み
df = pd.read_csv('../data/100stock_data.csv')
# date列を削除
df = df.drop(columns=['date'])
# 対数収益
prices = df.astype(float).values
logret = np.diff(np.log(prices), axis=0)        # [L-1, 100]
cols = list(df.columns)

target_col = 0
y   = logret[:, [target_col]]                   # ← ここを [] で2次元に（[L-1,1]）
exo = np.delete(logret, target_col, axis=1)     # [L-1, 99]

seq_single = np.concatenate([y, exo], axis=1)   # [L-1, 100]
data_single = [seq_single.astype(np.float32)]
D = data_single[0].shape[1]

data_all = []
for c in range(logret.shape[1]):
    y_c   = logret[:, [c]]
    exo_c = np.delete(logret, c, axis=1)
    seq_c = np.concatenate([y_c, exo_c], axis=1).astype(np.float32)  # [L-1, 100]
    data_all.append(seq_c)
# 使うモードを選ぶ：
use_all_targets = False
data = data_all if use_all_targets else data_single
D    = data[0].shape[1]

#データ分割
# logret: [T, N]  (T=時点数, N=銘柄数)
test_len = 200

def split_train_test_seq(seq_2d, test_len):
    # seq_2d: [L, D]（列0=ターゲット, 列1..=外生）
    L = seq_2d.shape[0]
    assert test_len < L
    return seq_2d[:L-test_len], seq_2d[L-test_len:]

# 例：単一ターゲット＋外生（D=100）
seq = seq_single.astype(np.float32)         # [L, D]
train_seq, test_seq = split_train_test_seq(seq, test_len)

# 複数系列（全銘柄を個別に学習したい場合）
train_data = []
test_data  = []
for seq in data_all:                        # 各seq: [L, D]
    tr, te = split_train_test_seq(seq, test_len)
    train_data.append(tr); test_data.append(te)

In [3]:
import itertools, random, copy
import numpy as np
import torch

def small_random_search(train_data, val_data, make_model, DreamingTrainer, trials=20, seed=42):
    random.seed(seed)
    best = None

    # 候補レンジ
    hidden_list = [64, 128, 192]
    layers_list = [1, 2]
    proj_list   = [None, 64, 128]
    dropout_list= [0.0, 0.1, 0.2]
    lr_list     = [1e-3, 5e-4]
    T_list      = [1.2, 1.5, 1.8]
    interleave  = ["epoch", "batch"]
    K_list      = [4, 6, 8]

    for t in range(trials):
        cfg = {
            "hidden": random.choice(hidden_list),
            "layers": random.choice(layers_list),
            "proj_dim": random.choice(proj_list),
            "dropout": random.choice(dropout_list),
            "lr": random.choice(lr_list),
            "T": random.choice(T_list),
            "mode": random.choice(interleave),
            "K": random.choice(K_list),
        }

        # モデル/最適化
        model = make_model(hidden_size=cfg["hidden"], num_layers=cfg["layers"],
                           proj_dim=cfg["proj_dim"], dropout=cfg["dropout"])
        opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=1e-6)

        trainer = DreamingTrainer(
            model=model, optimizer=opt, device="cuda" if torch.cuda.is_available() else "cpu",
            interleave_mode=cfg["mode"], K_interleave=cfg["K"],
            epochs=5,                   # 短いプローブ
            warm_vanilla_steps=200,
            max_vanilla_steps_per_epoch=min(200, len(train_data)),
            dreaming_steps_per_epoch=max(1, len(train_data)//5),
            dreaming_seq_len=50,
            temperature=cfg["T"], temperature_schedule="constant",
            grad_clip=1.0, logvar_clamp=(-12, 8), seed=seed+t
        )
        trainer.train(train_data, val_data=None)
        metrics = trainer.evaluate_metrics(data)  # {"nll","mse","mae"}
        score = metrics["nll"] + 0.1*np.sqrt(metrics["mse"])  # 例：NLL優先+RMSE補助

        print(f"[trial {t}] cfg={cfg} score={score:.4f} metrics={metrics}")
        if (best is None) or (score < best[0]):
            best = (score, copy.deepcopy(cfg), metrics)

    print("== BEST ==", best)
    return best


In [4]:
best = small_random_search(
    train_data=train_data, val_data=None,
    make_model=lambda hidden_size,num_layers,proj_dim,dropout:
        LSTMGaussian(input_dim=D, hidden_size=hidden_size,
                     num_layers=num_layers, proj_dim=proj_dim, dropout=dropout),
    DreamingTrainer=DreamingTrainer,
    trials=10, seed=123
)

[WARMUP] vanilla_steps=200
[WARMUP] step 50, loss=-1.390576, mse=0.006139, mae=0.057084
[VANILLA] (epoch mode) start
[VANILLA] step 50/98, loss=-3.337221, mse=0.000366, mae=0.013603
[DREAMING] (epoch mode) start, T=1.200
[EPOCH MODE] VANILLA avg_loss=-3.232515, DREAMING avg_loss=-2.701105, mse=0.000481, mae=0.015358
[EPOCH MODE] DREAMING avg_loss=-2.701105, mse=0.001183, mae=0.026514
[E1] SUMMARY: vanilla_loss=-3.232515, dreaming_loss=-2.701105, T=1.200
[VANILLA] (epoch mode) start
[VANILLA] step 50/98, loss=-3.171646, mse=0.000578, mae=0.017174
[DREAMING] (epoch mode) start, T=1.200
[EPOCH MODE] VANILLA avg_loss=-3.243437, DREAMING avg_loss=-2.892048, mse=0.000452, mae=0.014682
[EPOCH MODE] DREAMING avg_loss=-2.892048, mse=0.000710, mae=0.021121
[E2] SUMMARY: vanilla_loss=-3.243437, dreaming_loss=-2.892048, T=1.200
[VANILLA] (epoch mode) start
[VANILLA] step 50/98, loss=-3.446886, mse=0.000322, mae=0.011973
[DREAMING] (epoch mode) start, T=1.200
[EPOCH MODE] VANILLA avg_loss=-3.263219

In [ ]:
print("Best config:", best[1])
#best_params.jasonとしてresultsフォルダに保存
import json
with open("../results/best_params.json", "w") as f:
    json.dump(best[1], f, indent=4)

Best config: {'hidden': 64, 'layers': 1, 'proj_dim': 128, 'dropout': 0.2, 'lr': 0.001, 'T': 1.8, 'mode': 'batch', 'K': 8}
